### 1. (CREATE / INSERT) El equipo comercial quiere hacer un experimento rápido. Crea una tabla temporal (tb_reservas_prueba) y usa el comando INSERT con SELECT para insertar únicamente las reservas que corresponden a 'City Hotel' y que fueron canceladas (is_canceled = 1).


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tb_reservas_prueba AS
SELECT *
FROM cursodataacademy.hotel_db.tb_reservas
WHERE hotel = 'City Hotel'
    AND is_canceled = 1;

--CONCLUSION: Despues de realizar la consulta, la tabla quedo con 33102 reservas que fueron canceladas del City Hotel.

### 2. (UPDATE) Se detectó un error en los sistemas de captura de datos: los huéspedes que registraron babies en realidad no tenían bebés, sino que era un error del canal. Actualiza la tabla tb_reservas_prueba y cambia el valor de babies a 0 para todos los registros.


In [0]:
%sql
UPDATE workspace.default.tb_reservas_prueba
SET babies = 0;

--CONCLUSION: Se corrigieron 64 registros que presentaban babies > 0, aplicando el ajuste unicamente sobre la tabla temporal tb_reservas_prueba.

### 3. (CAST / CONCAT) Necesitamos tener la fecha de llegada en un formato legible. Selecciona las columnas de año, mes (que viene en texto, ej. 'July') y día, y concaténalas en una nueva columna llamada arrival_date_string con el formato "DD-Mes-YYYY" (Ej: "25-July-2015").

In [0]:
%sql
SELECT
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    CONCAT (
        LPAD(CAST(arrival_date_day_of_month AS STRING), 2, '0'), '-',
        arrival_date_month, '-',
        CAST (arrival_date_year AS STRING)
    ) AS arrival_date_string
FROM cursodataacademy.hotel_db.tb_reservas;

--CONCLUSION: La fecha se logra presentar en formato legible para su revisión (Ej: 19-October-2015).

### 4. (LIKE) Queremos filtrar a un segmento muy específico del canal de distribución. Selecciona todas las reservas donde distribution_channel empiece con la letra 'T'.

In [0]:
%sql
SELECT *
FROM cursodataacademy.hotel_db.tb_reservas
WHERE distribution_channel LIKE 'T%';

--CONCLUSION: Al filtrar con el patron 'T%', nos devuelve TA/TO como unico canal.

### 5. (COUNT / SUM / AVG) Calcula a nivel global para todo el hotel: la cantidad total de reservas, la suma total de solicitudes especiales (total_of_special_requests) y el promedio del precio por noche (adr).

In [0]:
%sql
SELECT
    COUNT (*) AS total_reservas,
    SUM (total_of_special_requests) AS total_solicitudes_especiales,
    ROUND(AVG(adr),2) AS adr_promedio
FROM cursodataacademy.hotel_db.tb_reservas;

--CONCLUSION: Se tiene un total de 119390 reservas y 68215 solicitudes especiales. Por otro lado, la tarifa promedio por noche asciende a USD 101.83.

### 6. (DELETE) En tu tabla de pruebas (tb_reservas_prueba), elimina todas las reservas que tuvieron un precio por noche (adr) igual a 0.0, ya que se consideran pruebas del sistema y no reservas reales.

In [0]:
%sql
DELETE FROM workspace.default.tb_reservas_prueba
WHERE adr = 0.0;

--CONCLUSION: Se elimino 129 registros con adr = 0.0.

### 7. (GROUP BY) El gerente quiere saber cuántas reservas entraron por cada canal de distribución. Obtén la cantidad de reservas agrupadas por distribution_channel.

In [0]:
%sql
SELECT
    distribution_channel,
    COUNT(*) AS total_reservas
FROM cursodataacademy.hotel_db.tb_reservas
GROUP BY distribution_channel
ORDER BY total_reservas DESC;

--CONCLUSION: El resultado nos presenta que TA/TO tiene un total de 96913 reservas, Direct tiene 13839 reservas, Corporate tiene 6485 reservas, GDS tiene 189 reservas y 5 reservas estan sin definir.

### 8. (CASE WHEN / Tipos de Datos) Crea una consulta que clasifique las reservas en base al precio (adr): Si es menor a 50 es "Económico", si está entre 50 y 150 es "Estándar", y mayor a 150 es "Premium".

In [0]:
%sql
SELECT
    adr,
    CASE
        WHEN adr < 50 THEN 'Economico'
        WHEN adr BETWEEN 50 AND 150 THEN 'Estandar'
        WHEN adr > 150 THEN 'Premium'
    END AS categoria_precio
FROM cursodataacademy.hotel_db.tb_reservas;

--CONCLUSION: Se clasifican las reservas en segmentos de precio, esto nos permitira diferenciar las ofertas economicas, estandar y premium.

### 9. (Funciones de Fecha - DATE_SUB / DATE_ADD) (Considera que existe una columna llamada arrival_date de tipo DATE para este ejercicio). Sabemos el lead_time (días de anticipación). Usa la función de Databricks para restar los días de lead_time a la arrival_date y así obtener la fecha exacta en la que el cliente hizo la reservación. Llámala booking_date.

In [0]:
%sql
--Se crea una tabla temporal para manipular las fechas de llegada y creacion de reserva.
CREATE OR REPLACE TEMP VIEW tv_reservas AS
SELECT
    *,
    TO_DATE(
    CONCAT_WS('-', arrival_date_year, arrival_date_month, arrival_date_day_of_month), 'yyyy-MMMM-d'
    ) AS arrival_date
FROM cursodataacademy.hotel_db.tb_reservas;

In [0]:
%sql
SELECT
    arrival_date,
    lead_time,
    DATE_SUB(
        arrival_date,
        CAST(lead_time AS INT)
    ) AS booking_date
FROM tv_reservas;

--CONCLUSION: Se reconstruyo la fecha de creación de cada reserva a partir de su llegada y anticipación, permitiendo pronosticar la demanda de estas fechas.

### 10. (Funciones Matemáticas) Queremos calcular una "cuota de riesgo". Selecciona el lead_time, divídelo entre 30 (para obtener meses aproximados) y usa la función CEIL() (redondeo hacia arriba) para mostrar el resultado en una columna llamada meses_anticipacion_techo.

In [0]:
%sql
SELECT
    lead_time,
    CEIL(lead_time/30) AS meses_anticipaci
FROM cursodataacademy.hotel_db.tb_reservas;

--CONCLUSION: Se expresa la anticipación meses, aproximando hacia el numero posterior. Con ello, facilitamos su clasificación.